# 05 · 损失函数模块（Losses）功能演示

演示 15 种自定义损失函数的梯度/二阶导、驱动 boosting 训练的效果对比、自定义评估指标与框架适配器。

In [1]:
import warnings, os
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import hscredit

# 路径约定：从 notebooks/ 目录运行，数据在 ../examples，产物输出到 model_report/
DATA = os.path.join("..", "examples", "hscredit_yyp.xlsx")
if not os.path.exists(DATA):
    DATA = os.path.join("examples", "hscredit_yyp.xlsx")
OUT = "model_report"
os.makedirs(OUT, exist_ok=True)

df = pd.read_excel(DATA)
df["放款时间"] = pd.to_datetime(df["放款时间"])
y = df["FPD"].astype(int)
NUM_FEATURES = ["珊瑚92", "青云24", "衡枢鉴真分老客版", "占信V3", "天创小额网贷分", "近六个月非银多头机构数"]
CAT_FEATURE = "商品类别"
print("数据形状:", df.shape)
print("坏样本率: {:.4f}".format(y.mean()))
df.head()

数据形状: (970, 18)
坏样本率: 0.1402


,客户编号,放款时间,放款金额,商品类别,MOB1,CURRENT_DPD,中智小牛分C3,珊瑚92,极光欺诈分6v1,青云24,占信V3,轻花老客海纳子分V1,天创小额网贷分,近六个月非银多头机构数,手机号近一个月非银多头机构数,身份证近一个月非银多头机构数,衡枢鉴真分老客版,FPD
0,1985945640026276096,2026-02-03,1399,礼包,0,0,NaN,NaN,NaN,656,NaN,NaN,630,51,15,15,0.0242,0
1,1985972188268592896,2026-02-04,1399,礼包,0,0,NaN,NaN,NaN,565,NaN,NaN,583,56,6,18,0.0492,0
2,1986034700861140992,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,708,NaN,NaN,764,68,17,20,0.0546,0
3,1986264852923760896,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,555,NaN,NaN,712,45,15,15,0.0899,0
4,1986265696509906944,2026-01-26,1399,礼包,0,0,NaN,NaN,NaN,581,NaN,NaN,641,67,32,32,0.0678,0


## 1. 15 种自定义损失函数 + 梯度/二阶导接口
面向风控的不平衡处理、成本敏感、排序/AUC、KS 分离、头部捕获、金额加权等。

In [2]:
from sklearn.model_selection import train_test_split
from hscredit.core.models.losses import (FocalLoss, AsymmetricFocalLoss, BalancedFocalLoss,
    WeightedBCELoss, CostSensitiveLoss, BadDebtLoss, ApprovalRateLoss, ProfitMaxLoss,
    ExpectedProfitLoss, OrdinalRankLoss, LiftFocusedLoss, RankingAUCProxyLoss, KSFocusedLoss,
    TopKBadCaptureLoss, AmountWeightedLoss, ExpectedValueLoss)

Xn = df[NUM_FEATURES].fillna(0).values
ytr_a = y.values
losses = {
    'FocalLoss': FocalLoss(), 'AsymmetricFocalLoss': AsymmetricFocalLoss(),
    'BalancedFocalLoss': BalancedFocalLoss(), 'WeightedBCELoss': WeightedBCELoss(auto_balance=True),
    'CostSensitiveLoss': CostSensitiveLoss(fn_cost=5.0), 'BadDebtLoss': BadDebtLoss(),
    'ApprovalRateLoss': ApprovalRateLoss(), 'ProfitMaxLoss': ProfitMaxLoss(),
    'ExpectedProfitLoss': ExpectedProfitLoss(), 'OrdinalRankLoss': OrdinalRankLoss(),
    'LiftFocusedLoss': LiftFocusedLoss(), 'RankingAUCProxyLoss': RankingAUCProxyLoss(),
    'KSFocusedLoss': KSFocusedLoss(), 'TopKBadCaptureLoss': TopKBadCaptureLoss(),
    'AmountWeightedLoss': AmountWeightedLoss(), 'ExpectedValueLoss': ExpectedValueLoss(),
}
probs = np.clip(np.random.RandomState(0).rand(len(ytr_a)), 1e-4, 1-1e-4)
rows = []
for name, loss in losses.items():
    g = loss.gradient(ytr_a.astype(float), probs)
    h = loss.hessian(ytr_a.astype(float), probs)
    rows.append({'损失函数': name, '梯度均值': round(float(np.mean(g)), 5),
                 '二阶导均值': round(float(np.mean(h)), 5) if h is not None else None})
pd.DataFrame(rows)

,损失函数,梯度均值,二阶导均值
0,FocalLoss,4.6653,1792.8619
1,AsymmetricFocalLoss,4.6680,1791.2107
2,BalancedFocalLoss,0.5667,449.9170
3,WeightedBCELoss,1.9250,2462.6558
4,CostSensitiveLoss,2.7671,2446.8167
5,BadDebtLoss,0.3539,0.1656
6,ApprovalRateLoss,0.3539,0.1656
7,ProfitMaxLoss,-0.3092,2.2619
8,ExpectedProfitLoss,-0.3776,10.5872
9,OrdinalRankLoss,0.3539,0.1661


## 2. 自定义损失驱动 XGBoost / LightGBM 训练对比

In [3]:
from hscredit.core.models import XGBoostRiskModel, LightGBMRiskModel
Xtr, Xte, ytr, yte = train_test_split(df[NUM_FEATURES].fillna(0), y, test_size=0.3, random_state=0, stratify=y)
rows = []
for name, loss in [('binary(基线)', None), ('FocalLoss', FocalLoss()), ('CostSensitiveLoss', CostSensitiveLoss(fn_cost=5.0)),
                   ('KSFocusedLoss', KSFocusedLoss()), ('TopKBadCaptureLoss', TopKBadCaptureLoss())]:
    kw = {} if loss is None else {'objective': loss}
    m = LightGBMRiskModel(n_estimators=80, **kw); m.fit(Xtr, ytr)
    ev = m.evaluate(Xte, yte)
    rows.append({'损失': name, 'KS': round(ev.get('ks', ev.get('KS', np.nan)), 4), 'AUC': round(ev.get('auc', ev.get('AUC', np.nan)), 4)})
loss_compare = pd.DataFrame(rows)
loss_compare

[1]	valid_0's binary_logloss: 0.401479
[2]	valid_0's binary_logloss: 0.397424
[3]	valid_0's binary_logloss: 0.396023
[4]	valid_0's binary_logloss: 0.396438
[5]	valid_0's binary_logloss: 0.393465
[6]	valid_0's binary_logloss: 0.392847
[7]	valid_0's binary_logloss: 0.392964
[8]	valid_0's binary_logloss: 0.388975
[9]	valid_0's binary_logloss: 0.393166
[10]	valid_0's binary_logloss: 0.392061
[11]	valid_0's binary_logloss: 0.390125
[12]	valid_0's binary_logloss: 0.393232
[13]	valid_0's binary_logloss: 0.397413
[14]	valid_0's binary_logloss: 0.397948
[15]	valid_0's binary_logloss: 0.399147
[16]	valid_0's binary_logloss: 0.402614
[17]	valid_0's binary_logloss: 0.405515
[18]	valid_0's binary_logloss: 0.407045
[19]	valid_0's binary_logloss: 0.406054
[20]	valid_0's binary_logloss: 0.405963
[21]	valid_0's binary_logloss: 0.410051
[22]	valid_0's binary_logloss: 0.414055
[23]	valid_0's binary_logloss: 0.416811
[24]	valid_0's binary_logloss: 0.418788
[25]	valid_0's binary_logloss: 0.422994
[26]	vali

[11]	valid_0's binary_logloss: 4.20797
[12]	valid_0's binary_logloss: 4.18152
[13]	valid_0's binary_logloss: 4.15628
[14]	valid_0's binary_logloss: 4.13546
[15]	valid_0's binary_logloss: 4.11394
[16]	valid_0's binary_logloss: 4.09575
[17]	valid_0's binary_logloss: 4.07939
[18]	valid_0's binary_logloss: 4.06664
[19]	valid_0's binary_logloss: 4.05215
[20]	valid_0's binary_logloss: 4.03741
[21]	valid_0's binary_logloss: 4.02855
[22]	valid_0's binary_logloss: 4.01618
[23]	valid_0's binary_logloss: 4.00621
[24]	valid_0's binary_logloss: 3.99775
[25]	valid_0's binary_logloss: 3.98609
[26]	valid_0's binary_logloss: 3.97902
[27]	valid_0's binary_logloss: 3.9676
[28]	valid_0's binary_logloss: 3.95705
[29]	valid_0's binary_logloss: 3.95155
[30]	valid_0's binary_logloss: 3.93776
[31]	valid_0's binary_logloss: 3.92867
[32]	valid_0's binary_logloss: 3.91923
[33]	valid_0's binary_logloss: 3.90937
[34]	valid_0's binary_logloss: 3.90433
[35]	valid_0's binary_logloss: 3.89662
[36]	valid_0's binary_logl

,损失,KS,AUC
0,binary(基线),0.2156,0.6183
1,FocalLoss,0.2254,0.6204
2,CostSensitiveLoss,0.3025,0.6652
3,KSFocusedLoss,0.1902,0.5965
4,TopKBadCaptureLoss,0.2890,0.6550


## 3. 自定义评估指标（KS / Gini / PSI）

In [4]:
from hscredit.core.models.losses import KSMetric, GiniMetric, PSIMetric
ks_m, gini_m = KSMetric(), GiniMetric()
print('KSMetric  :', round(float(ks_m(ytr.values.astype(float), m.predict_proba(Xtr)[:,1])), 4))
print('GiniMetric:', round(float(gini_m(ytr.values.astype(float), m.predict_proba(Xtr)[:,1])), 4))

KSMetric  : 0.4232
GiniMetric: 0.5106


## 4. 框架适配器（XGBoost / LightGBM / CatBoost 转换）

In [5]:
from hscredit.core.models.losses import XGBoostLossAdapter, LightGBMLossAdapter, CatBoostLossAdapter
loss = FocalLoss()
print('XGBoost objective callable:', callable(XGBoostLossAdapter(loss).objective()))
print('LightGBM objective callable:', callable(LightGBMLossAdapter(loss).objective()))
print('CatBoost loss object:', CatBoostLossAdapter(loss) is not None)

XGBoost objective callable: True
LightGBM objective callable: True
CatBoost loss object: True


## 5. 损失对比导出

In [6]:
loss_compare.to_excel(f"{OUT}/05_losses_compare.xlsx", index=False)
print('已保存损失函数对比')

已保存损失函数对比
